# MediEvil text generator

This notebook trains a trigram Markov model on `corpus.csv` and translates a prompt into MediEvil-style text. Run the cells in order to build the model and then generate new sentences from your own prompts.

In [54]:
from __future__ import annotations

from dataclasses import dataclass
from typing import List, Sequence, Tuple, Dict, Set, Optional

import pathlib
import re
import collections

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import spacy
import nltk
from nltk.corpus import wordnet as wn

In [55]:
# nltk.download("wordnet")
# nltk.download("omw-1.4")

In [57]:
nlp = spacy.load("en_core_web_lg")

In [58]:
def load_corpus_df(csv_path: str | pathlib.Path) -> pd.DataFrame:
    # load_corpus_df: Load MediEvil CSV into a DataFrame and validate columns.
    # Parameters:
    #   csv_path: Path to the CSV file.
    # Returns:
    #   Pandas DataFrame with non-empty Corpus rows.
    csv_path = pathlib.Path(csv_path)
    if not csv_path.is_file():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    df = pd.read_csv(csv_path, dtype=str).fillna("")
    required = {"Level", "Source", "Version", "Corpus"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {', '.join(sorted(missing))}")

    df["Corpus"] = df["Corpus"].str.strip()
    df = df[df["Corpus"] != ""].reset_index(drop=True)
    if df.empty:
        raise ValueError("No non-empty Corpus rows found in CSV")
    return df

In [59]:
def normalize_line(text: str) -> str:
    # normalize_line: Normalize a single text line.
    # Parameters:
    #   text: Original text string.
    # Returns:
    #   Lowercased string with standardized quotes and collapsed whitespace.
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

def extract_corpus_lines(
    df: pd.DataFrame,
    version: Optional[str] = "Final",
    normalized: bool = False,
) -> List[str]:
    # extract_corpus_lines: Extract corpus lines, optionally filtered and normalized.
    # Parameters:
    #   df: Full corpus DataFrame.
    #   version: Version filter (e.g. "Final") or None for all.
    #   normalized: Whether to normalize lines.
    # Returns:
    #   List of text lines.
    if version is not None:
        mask = df["Version"].str.strip().str.casefold() == version.casefold()
        df_use = df[mask].copy()
    else:
        df_use = df.copy()

    if df_use.empty:
        raise ValueError(f"No rows found for Version={version!r}")

    lines = [t.strip() for t in df_use["Corpus"].tolist() if t.strip()]
    if normalized:
        lines = [normalize_line(t) for t in lines]
    return lines


def analyze_corpus(lines: Sequence[str]) -> str:
    # analyze_corpus: Compute simple statistics over text lines.
    # Parameters:
    #   lines: Sequence of corpus lines.
    # Returns:
    #   Summary string with counts and length stats.
    if not lines:
        return "No lines in corpus."
    lengths = [len(line.split()) for line in lines]
    n = len(lines)
    avg_len = sum(lengths) / n
    max_len = max(lengths)
    return f"Lines: {n}, average length: {avg_len:.1f} tokens, max length: {max_len} tokens."

In [60]:
@dataclass
class MediEvilRetriever:
    # MediEvilRetriever: TF-IDF retriever over MediEvil lines.
    # Fields:
    #   lines: Original corpus lines.
    #   vectorizer: Fitted TfidfVectorizer.
    #   matrix: TF-IDF matrix for normalized lines.
    lines: List[str]
    vectorizer: TfidfVectorizer
    matrix: np.ndarray

    @classmethod
    def from_lines(cls, lines: Sequence[str]) -> "MediEvilRetriever":
        # from_lines: Build a retriever from raw lines.
        # Parameters:
        #   lines: Sequence of raw text lines.
        # Returns:
        #   MediEvilRetriever instance ready for similarity queries.
        raw_lines = [line.strip() for line in lines if line.strip()]
        if not raw_lines:
            raise ValueError("Cannot build retriever from empty lines")
        norm_lines = [normalize_line(line) for line in raw_lines]
        vectorizer = TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            stop_words="english",
        )
        matrix = vectorizer.fit_transform(norm_lines)
        return cls(lines=list(raw_lines), vectorizer=vectorizer, matrix=matrix)

    def most_similar(
        self,
        query: str,
        k: int = 5,
        min_score: float = 0.0,
    ) -> List[Tuple[str, float]]:
        # most_similar: Return top-k similar corpus lines for a query.
        # Parameters:
        #   query: Input text.
        #   k: Number of results to return.
        #   min_score: Minimum cosine similarity threshold.
        # Returns:
        #   List of (line, similarity) tuples sorted by similarity.
        if not self.lines:
            return []
        q_norm = normalize_line(query)
        q_vec = self.vectorizer.transform([q_norm])
        sims = cosine_similarity(q_vec, self.matrix)[0]
        k = max(1, min(k, len(self.lines)))
        idxs = np.argsort(-sims)[:k]
        results: List[Tuple[str, float]] = []
        for idx in idxs:
            score = float(sims[idx])
            if score < min_score:
                continue
            results.append((self.lines[idx], score))
        return results

In [61]:
def build_style_lexicon(lines: Sequence[str]) -> Dict[str, collections.Counter]:
    # build_style_lexicon: Build POS-based frequency lexicon from lines.
    # Parameters:
    #   lines: Corpus lines (raw text).
    # Returns:
    #   Dict of POS buckets ("noun", "verb", "adj", "adv", "other") to Counter.
    noun_freq = collections.Counter()
    verb_freq = collections.Counter()
    adj_freq = collections.Counter()
    adv_freq = collections.Counter()
    other_freq = collections.Counter()

    for line in lines:
        doc = nlp(line)
        for tok in doc:
            if not tok.is_alpha:
                continue
            lemma = tok.lemma_.lower()
            pos = tok.pos_
            if pos in {"NOUN", "PROPN"}:
                noun_freq[lemma] += 1
            elif pos == "VERB":
                verb_freq[lemma] += 1
            elif pos == "ADJ":
                adj_freq[lemma] += 1
            elif pos == "ADV":
                adv_freq[lemma] += 1
            else:
                other_freq[lemma] += 1

    return {
        "noun": noun_freq,
        "verb": verb_freq,
        "adj": adj_freq,
        "adv": adv_freq,
        "other": other_freq,
    }


def find_corpus_synonym(word: str, pos: str, lexicon: Dict[str, collections.Counter]) -> str:
    # find_corpus_synonym: Map a word to a synonym present in the MediEvil corpus.
    # Parameters:
    #   word: Original token text.
    #   pos: spaCy POS tag ("NOUN", "VERB", etc.).
    #   lexicon: Style lexicon from build_style_lexicon.
    # Returns:
    #   Either the best in-corpus synonym or the original word.
    pos_map = {
        "NOUN": "noun",
        "PROPN": "noun",
        "VERB": "verb",
        "ADJ": "adj",
        "ADV": "adv",
    }
    bucket = pos_map.get(pos, "other")
    freq_counter = lexicon[bucket]

    if word.lower() in freq_counter:
        return word

    if pos in {"NOUN", "PROPN"}:
        wn_pos = wn.NOUN
    elif pos == "VERB":
        wn_pos = wn.VERB
    elif pos == "ADJ":
        wn_pos = wn.ADJ
    elif pos == "ADV":
        wn_pos = wn.ADV
    else:
        wn_pos = None

    if wn_pos is None:
        return word

    synonyms: Set[str] = set()
    for syn in wn.synsets(word, pos=wn_pos):
        for lemma in syn.lemmas():
            lemma_name = lemma.name().replace("_", " ").lower()
            synonyms.add(lemma_name)

    candidates = [(s, freq_counter[s]) for s in synonyms if s in freq_counter]
    if not candidates:
        return word

    best = max(candidates, key=lambda x: x[1])[0]
    return best

In [62]:
GENERIC_STYLE_MAP: Dict[str, str] = {
    "you": "thou",
    "your": "thy",
    "yours": "thine",
    "yourself": "thyself",
    "do": "dost",
    "does": "doth",
    "did": "didst",
    "have": "hast",
    "has": "hath",
    "will": "shalt",
    "very": "most",
    "really": "most surely",
}

def medievilify(
    sentence: str,
    retriever: MediEvilRetriever,
    lexicon: Dict[str, collections.Counter],
    add_flourish: bool = True,
) -> str:
    # medievilify: Rewrite a sentence in MediEvil style while keeping meaning.
    # Parameters:
    #   sentence: Input sentence in plain English.
    #   retriever: MediEvilRetriever instance over the corpus.
    #   lexicon: Style lexicon built from corpus.
    #   add_flourish: Whether to append a short fragment from nearest in-game line.
    # Returns:
    #   Stylized sentence string.
    doc = nlp(sentence)
    out_tokens: List[str] = []

    for tok in doc:
        if tok.is_space:
            continue

        if tok.is_punct:
            out_tokens.append(tok.text)
            continue

        lower = tok.text.lower()

        if lower in GENERIC_STYLE_MAP:
            replacement = GENERIC_STYLE_MAP[lower]
            if tok.is_title:
                replacement = replacement.capitalize()
            out_tokens.append(replacement)
            continue

        if tok.pos_ in {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}:
            rep = find_corpus_synonym(tok.text, tok.pos_, lexicon)
            if tok.is_title:
                rep = rep.capitalize()
            out_tokens.append(rep)
            continue

        out_tokens.append(tok.text)

    text = ""
    for i, tok in enumerate(out_tokens):
        if i == 0:
            if tok:
                text += tok[0].upper() + tok[1:]
            else:
                text += tok
        elif tok in ".!,?;:":
            text += tok
        else:
            text += " " + tok

    if not text.endswith((".", "!", "?")):
        text += "."

    if add_flourish:
        neighbours = retriever.most_similar(sentence, k=1)
        if neighbours:
            best_line, score = neighbours[0]
            parts = best_line.split(",")
            tail = parts[-1].strip()
            if 4 <= len(tail.split()) <= 14:
                if not text.endswith((".", "!", "?")):
                    text += "."
                text += " " + tail

    return text

In [63]:
csv_file = "corpus.csv"

df = load_corpus_df(csv_file)
corpus_lines_raw = extract_corpus_lines(df, version="Final", normalized=False)
print(analyze_corpus(corpus_lines_raw))

retriever = MediEvilRetriever.from_lines(corpus_lines_raw)
style_lexicon = build_style_lexicon(corpus_lines_raw)

test_sentence = "I speedrun the entire game"
print("INPUT: ", test_sentence)
print("OUTPUT:", medievilify(test_sentence, retriever, style_lexicon))

Lines: 216, average length: 33.8 tokens, max length: 553 tokens.
INPUT:  I speedrun the entire game
OUTPUT: I speedrun the full plot.
